In [ ]:
import sys
import cv2
import torch
import numpy as np
from pathlib import Path
RAW_DIR = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/exports/export-4-10/extracted/images")
OUTPUT_VIDEO = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/exports/export-4-10/guideline_overlay.mp4")
CHECKPOINT = "checkpoints/cardio/cardio_run/fold1/best_fea2.pth"
IMG_SIZE = 224
MAX_FRAMES = 1000
FPS = 18.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Developer config — edit only this block when switching features/checkpoints.
# Class index i must match logit i from the model head. If masks look empty,
# check len(CLASS_NAMES) == NUM_MODEL_CLASSES after the model cell, or set
# MODEL_CLASS_TO_SEMANTIC to a uint8 LUT of length NUM_MODEL_CLASSES.
#
# label_fea3_p10 (+ legacy 1–3): TAG_TO_CLASS maps tag string -> model id (0 = background).
# ---------------------------------------------------------------------------
MODEL_CLASS_TO_SEMANTIC = None  # or np.array([...], dtype=np.uint8) length NUM_MODEL_CLASSES

TAG_TO_CLASS = {
    "epicardial adipose tissue": 1,
    "pericardium": 2,
    "phrenic nerve": 3,
    "aortic root": 4,
    "auricles": 5,
    "epicardial fat on aortic": 6,
    "grasper": 7,
    "needle holders": 8,
}
# CLASS_NAMES[i] must match logit i (9 classes: 0..8).
CLASS_NAMES = [
    "Background",
    "Epicardial adipose tissue",
    "Pericardium",
    "Phrenic nerve",
    "Aortic root",
    "Auricles",
    "Epicardial fat on aortic",
    "Grasper",
    "Needle holders",
    "Pericardial stay sutures",
    "Pericardium boundary"
]
CLASS_COLORS_BGR = [
    (0, 0, 0),
    (0, 165, 255),
    (0, 200, 0),
    (255, 0, 0),
    (255, 128, 0),
    (255, 0, 255),
    (0, 255, 255),
    (180, 105, 255),
    (42, 42, 165),
    (255, 0, 255),
    (0, 255, 0),

]
ALL_FOREGROUND_IDS = tuple(range(1, len(CLASS_NAMES)))

GUIDELINE_ANCHOR_CLASS_ID = TAG_TO_CLASS["pericardium"]  # 2 — centerline / band
OVERLAY_ALPHA_CLASS_IDS = tuple(c for c in ALL_FOREGROUND_IDS if c != GUIDELINE_ANCHOR_CLASS_ID)
CALLOUT_CLASS_IDS = ALL_FOREGROUND_IDS
DISPLAY_CLASS_NAMES = {}  # optional {class_id: "short label"}
BLINK_WARNING_CLASS_ID = TAG_TO_CLASS["phrenic nerve"]  # 3 — or None
BLINK_WARNING_PERIOD_FRAMES = 8

EXPORT_MASK_LAYERS = [
    {
        "class_id": 1,
        "subdir": "class01_epicardial_adipose",
        "file_prefix": "class01_",
        "json_key": "class_1_epicardial_adipose_tissue",
        "note": "0/255 binary, model id 1 (epicardial adipose tissue).",
    },
    {
        "class_id": 2,
        "subdir": "class02_pericardium",
        "file_prefix": "class02_",
        "json_key": "class_2_pericardium_full_segmentation",
        "note": "0/255 binary, model id 2 (pericardium); guideline anchor.",
    },
    {
        "class_id": 3,
        "subdir": "class03_phrenic",
        "file_prefix": "class03_",
        "json_key": "class_3_phrenic_nerve",
        "note": "0/255 binary, model id 3 (phrenic nerve).",
    },
    {
        "class_id": 4,
        "subdir": "class04_aortic_root",
        "file_prefix": "class04_",
        "json_key": "class_4_aortic_root",
        "note": "0/255 binary, model id 4 (aortic root).",
    },
    {
        "class_id": 5,
        "subdir": "class05_auricles",
        "file_prefix": "class05_",
        "json_key": "class_5_auricles",
        "note": "0/255 binary, model id 5 (auricles).",
    },
    {
        "class_id": 6,
        "subdir": "class06_epicardial_fat_aortic",
        "file_prefix": "class06_",
        "json_key": "class_6_epicardial_fat_on_aortic",
        "note": "0/255 binary, model id 6 (epicardial fat on aortic).",
    },
    {
        "class_id": 7,
        "subdir": "class07_grasper",
        "file_prefix": "class07_",
        "json_key": "class_7_grasper",
        "note": "0/255 binary, model id 7 (grasper).",
    },
    {
        "class_id": 8,
        "subdir": "class08_needle_holders",
        "file_prefix": "class08_",
        "json_key": "class_8_needle_holders",
        "note": "0/255 binary, model id 8 (needle holders).",
    },
]

# Downstream JSON keeps key "pericardium_class"; value is the guideline anchor id.
PERICARDIUM_CLASS = GUIDELINE_ANCHOR_CLASS_ID

ALPHA = 0.5
GUIDELINE_COLOR_BGR = (0, 255, 0)
OFFSET_CM = 1.2
PIXELS_PER_CM = 20.0
OFFSET_COLOR_POS_BGR = (0, 140, 255)
OFFSET_COLOR_NEG_BGR = (255, 0, 0)

DASH_LEN = 12
GAP_LEN = 8
LINE_THICKNESS = 2
OFFSET_LINE_THICKNESS = 2

TV_WEIGHT = 0.15
TEMPORAL_ALPHA = 0.25
MAX_CENTERLINE_JUMP_PX = 25

BAND_WIDTH_SCALE = 12.0
BAND_ALPHA = 0.18
BAND_COLOR_BGR = (0, 255, 0)

CALLOUT_TEXT_COLOR = (255, 255, 255)
CALLOUT_BG_COLOR = (20, 20, 20)
CALLOUT_ARROW_COLOR = (255, 255, 255)
CALLOUT_FONT_SCALE = 0.55
CALLOUT_THICKNESS = 1
CALLOUT_PADDING = 6
CALLOUT_MIN_MASK_PIXELS = 1
CALLOUT_OFFSET_CM = 2.0

DANGER_TRIANGLE_COLOR = (0, 0, 255)
DANGER_TRIANGLE_OUTLINE = (255, 255, 255)

from guideline_export_config import configure_notebook_overlay

configure_notebook_overlay(DEVICE, IMG_SIZE)


In [ ]:
import numpy as np
import torch
from Models.DeepLabV3Plus.modeling import deeplabv3plus_resnet101

from checkpoint_loading import infer_num_classes, load_checkpoint_state
from guideline_export_config import (
    assert_class_names_match_checkpoint,
    build_model_pred_to_semantic,
)

state = load_checkpoint_state(CHECKPOINT, map_location=DEVICE)
NUM_MODEL_CLASSES = infer_num_classes(state)

model = deeplabv3plus_resnet101(
    num_classes=NUM_MODEL_CLASSES, output_stride=8, pretrained_backbone=False
)
model.load_state_dict(state, strict=True)
model = model.to(DEVICE)
model.eval()

_lut = None if MODEL_CLASS_TO_SEMANTIC is None else np.asarray(MODEL_CLASS_TO_SEMANTIC, dtype=np.uint8)
model_pred_to_semantic = build_model_pred_to_semantic(NUM_MODEL_CLASSES, CLASS_NAMES, _lut)
assert_class_names_match_checkpoint(NUM_MODEL_CLASSES, CLASS_NAMES, _lut)

print(f"Model loaded (num_classes={NUM_MODEL_CLASSES}).")


In [ ]:
from guideline_overlay_core import overlay_guideline, preprocess_frame


In [ ]:
all_paths = sorted(RAW_DIR.glob("frame_*.jpg"), key=lambda p: int(p.stem.split("_")[1]))
frame_paths = all_paths[:MAX_FRAMES]
print(f"Dùng {len(frame_paths)} frame đầu (tối đa {MAX_FRAMES}). Tổng trong raw: {len(all_paths)}")
if not frame_paths:
    raise SystemExit("Không có frame_*.jpg trong raw.")

In [ ]:
first = cv2.imread(str(frame_paths[0]))
if first is None:
    raise SystemExit(f"Không đọc được: {frame_paths[0]}")
h, w = first.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, FPS, (w, h))
print(f"Output video: {OUTPUT_VIDEO}, {w}x{h}, {FPS} fps")

In [ ]:
from feature_zip_export import run_frames_guideline_pipeline

run_frames_guideline_pipeline(
    frame_paths,
    model,
    model_pred_to_semantic,
    video_writer=out,
    progress_every=100,
    video_progress_message="Processed",
)
out.release()
print("Done video:", OUTPUT_VIDEO)


In [ ]:
# Task 2 — same layout as overlay_video_segment.py --export (frames/, masks/, json/, zip).
from pathlib import Path

from feature_zip_export import run_task2_feature_zip_export

OUTPUT_SEQ_DIR = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/features_data/")
OUTPUT_SEQ_ZIP = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/features_data/feature_2_data.zip")
MAX_EXPORT_FRAMES = 1000

run_task2_feature_zip_export(
    raw_dir=RAW_DIR,
    frame_paths=None,
    model=model,
    model_pred_to_semantic=model_pred_to_semantic,
    output_seq_dir=OUTPUT_SEQ_DIR,
    output_seq_zip=OUTPUT_SEQ_ZIP,
    max_frames=MAX_EXPORT_FRAMES,
    clean_export=True,
    mask_as_bgr=True,
    write_zip=True,
    progress_every=100,
    export_mask_layers=EXPORT_MASK_LAYERS,
    class_names=CLASS_NAMES,
    guideline_anchor_class_id=GUIDELINE_ANCHOR_CLASS_ID,
)
